In [14]:
# Cell 1: Imports and Parameters
import cv2, numpy as np
from datetime import datetime

# HSV thresholds
oil_lower = np.array([95, 60, 160], dtype=np.uint8)
oil_upper = np.array([115,255,255], dtype=np.uint8)
tm_lower  = np.array([10, 60, 180], dtype=np.uint8)
tm_upper  = np.array([30, 255, 255], dtype=np.uint8)
cool_lower= np.array([75, 30, 180], dtype=np.uint8)
cool_upper= np.array([95, 120, 255], dtype=np.uint8)

# Exclusion masks
sticker_low = np.array([25, 50, 60], dtype=np.uint8)
sticker_high= np.array([60, 255,255], dtype=np.uint8)
violet_low  = np.array([115,30,50], dtype=np.uint8)
violet_high = np.array([165,255,255], dtype=np.uint8)

# Morphological params
kernel = np.ones((5,5), np.uint8)
min_area = 800  # px^2 (tune as needed)
min_aspect, max_aspect = 0.15, 8.0  # allowable blob shape


In [15]:
# Cell 2: Color Conversion Functions and Tests
def hsl_to_hsv(h_deg, s_frac, l_frac):
    """
    Convert HSL (H [0,360], s,l [0,1]) to OpenCV HSV (H [0,180], S,V [0,255]).
    Formula: V=L+S_L*min(L,1-L); S_V=2*(1-L/V) if V>0【45†L1369-L1373】.
    """
    H = h_deg % 360
    L = l_frac; S_L = s_frac
    V = L + S_L*min(L, 1-L)
    if V == 0:
        S_V = 0
    else:
        S_V = 2*(1 - L/V)
    H_cv = H/2
    S_cv = S_V * 255
    V_cv = V * 255
    return (int(round(H_cv)), int(round(S_cv)), int(round(V_cv)))

# Sample conversions and verification
print("Engine oil (#85b5f6):", cv2.cvtColor(np.uint8([[[246,181,133]]]), cv2.COLOR_BGR2HSV)[0,0])
print("TM Oil HSL->HSV:", hsl_to_hsv(37, 0.47, 0.70))
print("Coolant HSL->HSV:", hsl_to_hsv(167, 0.33, 0.75))


Engine oil (#85b5f6): [107 117 246]
TM Oil HSL->HSV: (18, 86, 214)
Coolant HSL->HSV: (84, 51, 212)


In [16]:
# Cell 3: Preprocessing (Gray-world WB and CLAHE)
def apply_grayworld(img):
    """
    Simple gray-world white-balance: scale each channel so its average equals
    the average of all channels.
    """
    b,g,r = cv2.split(img.astype(np.float32))
    avgR = np.mean(r); avgG = np.mean(g); avgB = np.mean(b)
    avg = (avgR+avgG+avgB)/3
    r = r * (avg/avgR)
    g = g * (avg/avgG)
    b = b * (avg/avgB)
    balanced = cv2.merge([b,g,r]).astype(np.uint8)
    return balanced

def preprocess(img):
    """
    Apply white balance (gray-world) and CLAHE on V channel.
    """
    wb = apply_grayworld(img)
    hsv = cv2.cvtColor(wb, cv2.COLOR_BGR2HSV)
    h,s,v = cv2.split(hsv)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    v = clahe.apply(v)
    hsv_eq = cv2.merge([h,s,v])
    return cv2.cvtColor(hsv_eq, cv2.COLOR_HSV2BGR)

# Example: Apply to an input image (placeholder 'input.jpg')
input_img = cv2.imread('input.jpeg')  # Replace with actual path
img_proc = preprocess(input_img)


In [ ]:
# Cell 4: Detect Function for One Image
def detect_frame(frame):
    """
    Detect and annotate leaks in a single UV image.
    Returns annotated image, binary masks, and leak counts.
    """
    annotated = frame.copy()
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    # Masks for each fluid
    mask_oil  = cv2.inRange(hsv, oil_lower, oil_upper)
    mask_tm   = cv2.inRange(hsv, tm_lower,  tm_upper)
    mask_cool = cv2.inRange(hsv, cool_lower, cool_upper)
    # Exclusions
    excl_st = cv2.inRange(hsv, sticker_low, sticker_high)
    excl_vi = cv2.inRange(hsv, violet_low, violet_high)
    mask_oil  = cv2.bitwise_and(mask_oil,  cv2.bitwise_not(excl_st))
    mask_tm   = cv2.bitwise_and(mask_tm,   cv2.bitwise_not(excl_st))
    mask_oil  = cv2.bitwise_and(mask_oil,  cv2.bitwise_not(excl_vi))
    mask_tm   = cv2.bitwise_and(mask_tm,   cv2.bitwise_not(excl_vi))
    mask_cool = cv2.bitwise_and(mask_cool, cv2.bitwise_not(excl_vi))
    # Morphology cleanup
    for m in [mask_oil, mask_tm, mask_cool]:
        cv2.morphologyEx(m, cv2.MORPH_CLOSE, kernel, iterations=2, dst=m)
        cv2.morphologyEx(m, cv2.MORPH_OPEN,  kernel, iterations=1, dst=m)
    leak_counts = {"OIL":0, "TM":0, "COOL":0}
    colors = {"OIL":(0,0,255), "TM":(0,255,255), "COOL":(255,255,0)}
    # Analyze contours
    for label, mask in zip(["OIL","TM","COOL"], [mask_oil, mask_tm, mask_cool]):
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for cnt in contours:
            area = cv2.contourArea(cnt)
            if area < min_area: continue
            x,y,w,h = cv2.boundingRect(cnt)
            ar = w/float(h) if h>0 else 0
            if ar < min_aspect or ar > max_aspect: continue
            leak_counts[label] += 1
            cv2.rectangle(annotated, (x,y), (x+w, y+h), colors[label], 2)
            cv2.putText(annotated, f"{label} {int(area)}px", (x, y-6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, colors[label], 1)
    # Status text
    any_leak = any(v>0 for v in leak_counts.values())
    status = "LEAK DETECTED" if any_leak else "ALL CLEAR"
    color = (0,0,255) if any_leak else (0,200,0)
    cv2.putText(annotated, status, (10,30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    cv2.putText(annotated, ts, (10, frame.shape[0]-10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200,200,200), 1)
    return annotated, mask_oil, mask_tm, mask_cool, leak_counts

# Cell 5: Run Detection on an Input Image
annotated, m_oil, m_tm, m_cool, counts = detect_frame(img_proc)
print("Leak counts:", counts)
# Display results
cv2.imshow("Annotated", annotated)
cv2.imshow("Oil Mask", m_oil)
cv2.imshow("TM Mask", m_tm)
cv2.imshow("Cool Mask", m_cool)
cv2.waitKey(0)
cv2.destroyAllWindows()


Leak counts: {'OIL': 0, 'TM': 1, 'COOL': 0}


In [ ]:
# Cell 6: Calibration - ROI Color Picker
samples = []
def pick_color(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        hsv_val = hsv_img[y,x]
        samples.append(hsv_val)
        print("Picked HSV:", hsv_val)
    if event == cv2.EVENT_RBUTTONDOWN:
        # Compute k-means cluster on samples
        if samples:
            pts = np.float32(samples)
            _,_,center = cv2.kmeans(pts, 1, None,
                                   (cv2.TERM_CRITERIA_EPS, 10, 1.0), 
                                   3, cv2.KMEANS_PP_CENTERS)
            center = center[0]
            low = np.maximum(center - 20, 0)
            high= np.minimum(center + 20, [180,255,255])
            print("Auto-tuned HSV range:", low.astype(int), high.astype(int))
        cv2.destroyWindow("Pick Sample HSV")

# Example usage:
hsv_img = cv2.cvtColor(img_proc, cv2.COLOR_BGR2HSV)
cv2.namedWindow("Pick Sample HSV")
cv2.setMouseCallback("Pick Sample HSV", pick_color)
cv2.imshow("Pick Sample HSV", img_proc)
cv2.waitKey(0)


Picked HSV: [ 70   4 188]
Picked HSV: [ 25  81 133]
Picked HSV: [ 23  73 144]


-1

In [ ]:
# Cell 7: Unit Tests (FIXED SIMPLE VERSION)

# ---- OIL TEST ----
patch_oil = np.zeros((50,50,3), np.uint8)
patch_oil[:] = (246,181,133)  # known BGR
_, mo, mt, mc, cnts_oil = detect_frame(patch_oil)
assert cnts_oil["OIL"] >= 1, "Oil patch not detected!"


# ---- TM OIL TEST ----
patch_tm = np.zeros((50,50,3), np.uint8)

tm_hsv = np.uint8([[[18, 85, 214]]])   # HSV
tm_bgr = cv2.cvtColor(tm_hsv, cv2.COLOR_HSV2BGR)[0,0]

patch_tm[:] = tm_bgr

_, mo, mt, mc, cnts_tm = detect_frame(patch_tm)
assert cnts_tm["TM"] >= 1, "TM patch not detected!"


# ---- COOLANT TEST ----
patch_cool = np.zeros((50,50,3), np.uint8)

cool_hsv = np.uint8([[[84, 51, 212]]])  # HSV
cool_bgr = cv2.cvtColor(cool_hsv, cv2.COLOR_HSV2BGR)[0,0]

patch_cool[:] = cool_bgr

_, mo, mt, mc, cnts_cool = detect_frame(patch_cool)
assert cnts_cool["COOL"] >= 1, "Coolant patch not detected!"


print("✅ Unit tests passed!")

✅ Unit tests passed!
